<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/disinfection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Disinfection calculation***


In [ ]:
import math as mh
import pandas as pd

def disin(lanes,Partable2,dailyflow,Treatment):
    # Disinfection  design
    # This code calculates the dimensions of disinfection stage in a water treatment plant.
    #treatmenttechnologies2 = treatmenttechnologies.astype(str)


    # Input parameters
    print("")
    print("DISINFECTION PROCESS")
    flowRate = dailyflow/24  # m3/h
    enter = mh.ceil(Partable2['Enterococci'][2]*10)/10
    print("The average maximum measured Enterococci (col/100ml) in the source water is",enter)
    ecoli = mh.ceil(Partable2['Escherichia coli (E.coli)'][2]*10)/10
    print("The average maximum measured Escherichia coli (col/100ml) in the source water is",ecoli)

    # Calculate surface loading rate
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "chlorination").any():
      disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth=cl(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "chloramination").any():
      disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth=clm(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "lp-uv").any():
      disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth=lpuv(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "ozonation").any():
      disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth=ozon(flowRate,lanes) #, units
    else:
      disarea = 0
      disunitarea = 0
      ompunit = 0
      disstorage = 0
      dosageflow = 0
      dosageflow2 = 0
      disheadloss = 0
      Cldaily = 0
      ammdaily = 0
      Edis = 0
    return disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth

def cl(flowRate,lanes):
    flowlane=flowRate/lanes # m3/h
    tbr= 30  # retention time in minutes
    print(f'The retention time is assumed equal to {tbr} minutes')
    clvolume=mh.ceil(flowlane*tbr/60) #m3
    print(f'The volume of the chlorination tank (one per lane) is {clvolume}m3')
    D=float(input("Set the depth of the chlorination tank (m)"))
    disunitarea=mh.ceil(clvolume/D) #m2
    print(f'The area of the  chlorination tank per lane is {disunitarea}m2')
    disarea=mh.ceil(disunitarea*lanes)
    print(f'The total area of the  chlorination process is {disarea}m2')
    Clres=float(input("The desired free chlorine residual (mg/l)is (default is 0.2mg/l):") or 0.2)
    Cldos=round(Clres*3/0.5,1) # dosage in mg/l
    print('The chlorine dosage (mg/l) is',Cldos)
    Cldaily=mh.ceil(lanes*flowlane*Cldos*lanes*1000*24/1000000) #kg/day
    Cldaily=mh.ceil(Cldaily/0.15) #kg/day with a solution of 1/15
    print(f'Assuming 15% chlorine solution,the total daily chlorine requirements are {Cldaily}kg/d',)
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1 day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    Clmonthly=mh.ceil(Cldaily*ndays) #kg/month
    disvolume=(Clmonthly/1000) #1% solution mg/l
    disstorage=mh.ceil(disvolume/2.5) # 2.5m of height.
    print(f' The required area for {ndays} days storage of chlorine solutiosn is {disstorage}m2')
    dosageflow=round(Cldaily*1000/(1000*24)) #/l/h
    print(f'The chlorine dosage pump nominal flow  is {dosageflow}l/h')
    dosageflow2=0
    ammdaily=0
    Pdis=0.1 #KW
    Edis=mh.ceil(Pdis*24) #KWh
    print(f'The daily energy consumption of the chlorine dosage pumps is {Edis}kWh')
    L=3*mh.sqrt(disarea/3)
    disheadloss=round(L*0.01+D,2)
    disdepth=D
    return disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily, ammdaily,disdepth

def clm(flowRate,lanes):
    flowlane=flowRate/lanes # m3/h
    tbr= 30  # retention time in minutes
    print(f'The retention time is assumed equal to {tbr} minutes')
    clmvolume=mh.ceil(flowlane*tbr/60) #m3
    print('The volume of the  chloramination tank (m3) is',clmvolume)
    D=float(input("Set the depth of the chloramination tank (m)"))
    disunitarea=mh.ceil(clmvolume/D) #m2
    print(f'The area of the  chloramination tank per lane is {disunitarea}m2')
    disarea=disunitarea*lanes
    print(f'The total area of the chloramination process is {disarea}m2')
    print("For chloramination, ammonia and chlorine are added in a ratio of 1:4 (ammonia) to achieve a residual chloramine level of around 2 mg/L")
    Clmdos=3 # dosage in mg/l
    print(f'The chlorine dosage is {Clmdos}mg/L')
    Cldaily=mh.ceil(lanes*flowlane*Clmdos*lanes*24/1000) #kg/day
    Cldaily=mh.ceil(Cldaily/15) #kg/day with a solution of 1/15
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1 day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    Clmonthly=Cldaily*ndays #kg/days storage
    ammon=mh.ceil(Clmonthly/4) #kg for storage period
    ammdaily=mh.ceil(ammon/ndays) #kg/day
    disvolume=((Clmonthly+ammon))/1000
    disstorage=mh.ceil(disvolume/2.5)# 2.5m of height.
    print(f' The required area for {ndays} days storage of chlorine solutios is {disstorage}m2')
    dosageflow=mh.ceil(Cldaily*1000/(1000*24)) #/l/h
    print(f'The chlorine dosage pump nominal flow is {dosageflow}l/h')
    dosageflow2=mh.ceil(Cldaily/4) #/l/h
    print(f'The ammonia dosage pump nominal flow is {dosageflow2}l/h')
    Pdis=0.2 #KW
    Edis=mh.ceil(Pdis*24) #KW
    print(f'The daily energy consumption of the two dosage pumps (chlorine dosage pump and ammonia dosage pump) is {Edis}kW')
    L=3*mh.sqrt(disarea/3)
    disheadloss=round(L*0.01+D,2)
    disdepth=D
    return disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth

def ozon(flowRate,lanes):
    while True:
        tbr = input("Set the required retention time (min 8min, max 20min, default 15):") or "15"
        try:
            tbr = int(tbr)
            if 8 <= tbr <= 20:
                break
            else:
                print("Invalid input. Enter a number between 8 and 20.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    while True:
        units_str = input("Number of different ozonation units per lane:")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number greater than or equal to 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    flowunit=flowRate/(lanes*units) # m3/h
    print('The Ozone dosage for disinfection only is assumed to be 2 mg/l')
    O3=2 #mg/l
    ozonevolume=flowunit*tbr/60
    D=float(input("Set the depth of the ozonation tank (m)"))
    disunitarea=mh.ceil(ozonevolume/ozonedepth) #m2
    disarea=mh.ceil(disunitarea*lanes*units)
    print(f"The area of the ozonation unit is {disunitarea}m2")
    print(f"The total area of the ozonation process tanks is {disarea}m2")
    ozonedem=O3*flowRate*0.001 #kg/h
    ozonedem=round(ozonedem*1.1,2) #safety factor
    print(f"The ozone demand is {ozonedem}kg/h")
    ozgensize=mh.ceil(ozonedem*10) #10m2 per kg produced
    print("The area required for the the ozone generators (m2):",ozgensize)
    disarea=mh.ceil(disarea+ozgensize)
    print("The total area of the ozonation process (ozonation tank and storage of ozonators) (m2):",disarea)
    disstorage=0
    dosageflow=0
    dosageflow2=0
    Cldaily=0
    ammdaily=0
    eo3=15 # 15 kWh/kgO3
    Edis=mh.ceil(eo3*ozonedem*24) #KW
    print(f"The daily energy consumption per hour required for the ozonation is {Edis}KWh")
    L=3*mh.sqrt(disarea/3)
    disheadloss=round(L*0.01+D,2)
    disdepth = D
    return disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily,disdepth

def lpuv(flowRate,lanes):
    flowlane=flowRate/lanes # m3/h
    UVdose=int(input("Set the low pressure UV dose(typically 30-70 mJ/cm2):"))
    UVint=int(input('Set the UV intensity(typically 30-40 mW/cm² for low pressure UV lamps):'))
    tbr= UVdose/UVint #in min
    print(f'The exposure time is equal to {tbr} sec')
    uvvolume= flowlane*tbr/60 #m3


    while True:
        D=float(input("Set the depth of the UV disnfection tank in meters (min 0.6m - max 2.0m - default 1m)")or 1)
        try:
            if  0.6<= D <= 2:
                break
            else:
                print("Invalid input. Enter a number between 0.6 and 2.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")
    disunitarea=uvvolume/D #m2
    disarea=mh.ceil(disunitarea*lanes)
    print(f"The total area of the LP-UV disinfection is{disarea}m2")
    Pdis=mh.ceil(6*disarea*UVint*10/1000) #KW
    print("The power consumption required for the LP-UV(KW) is",Pdis)
    Edis=Pdis*24 #KW
    Edis=mh.ceil(Edis)
    print(f'The daily energy consumption of the LP-UV process is {Edis}kW')
    disstorage=0
    dosageflow=0
    dosageflow2=0
    Cldaily=0
    ammdaily=0
    L=3*mh.sqrt(disarea/3)
    disheadloss=round(L*0.01+D,2)
    disdepth = D
    return disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily, disdepth